[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/09_mcp-multi-agent-assistant.ipynb)

*Industry Project 9 of 10 · AI Engineer · Advanced · about 8–10 hours*

# Multi-Agent Support Analyst over an MCP Server

**The situation.** *ShopKart*'s head of customer support asks every Monday: *"What are customers complaining about, which issues are most urgent, and where should engineering look first?"* Today an analyst spends half a day reading tickets.

You'll build an AI system that answers it, the way modern AI products are built:

1. An **MCP server** that exposes the ticket database as tools any AI host can use (Claude Desktop, Claude Code, your own agent).
2. An **MCP client** that talks to it over JSON-RPC.
3. A **multi-agent** pipeline: an orchestrator fans out to research agents, a writer drafts the report with citations, and a reviewer rejects any claim without a source.
4. **Tracing** of every tool call, so you can see what the agents did.

Everything runs offline with deterministic stand-ins for the model; the last section swaps in a real Claude agent loop if you have an API key.

**What you'll hand in:** an MCP server (tools/list and tools/call over JSON-RPC), a client, an orchestrator with research, writer and reviewer agents, a cited Monday report, and a trace summary — plus the same server packaged as a real MCP server file.

**Skills:** MCP, JSON-RPC, tool schemas, multi-agent orchestration, citations, reviewer agents, tracing, Claude API tool use

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**support_tickets.csv** — 240 tickets from the last 60 days:

| Column | Meaning |
|---|---|
| ticket_id | T001 … T240 |
| product | App, Web, Payments, Delivery |
| category | bug, billing, delivery, login |
| priority | low, medium, high |
| status | open, closed |
| day | 1 … 60 |
| text | the customer's issue, in short |

The tools below are given — your job is the protocol, the agents and the checks around them.

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return
    print(f"❌ {name}: not yet — {_why(got, want)}")

import inspect, json, re, time
from collections import Counter
tickets = load("support_tickets.csv")


def search_tickets(query: str, limit: int = 5) -> str:
    """Search open tickets by keywords; most matching words first."""
    words = set(re.findall(r"[a-z]+", query.lower()))
    scored = []
    for row in tickets[tickets.status == "open"].itertuples():
        hits = len(words & set(re.findall(r"[a-z]+", row.text.lower() + " " + row.category + " " + row.product.lower())))
        if hits:
            scored.append((-hits, row.ticket_id, row))
    scored.sort()
    return json.dumps([{"id": r.ticket_id, "product": r.product, "priority": r.priority, "text": r.text} for _, _, r in scored[:limit]])


def ticket_counts(field: str) -> str:
    """Count open tickets by a field: product, category or priority."""
    if field not in ("product", "category", "priority"):
        raise ValueError(f"unknown field: {field}")
    return json.dumps(tickets[tickets.status == "open"][field].value_counts().sort_index().to_dict())


def get_ticket(ticket_id: str) -> str:
    """Full details of one ticket."""
    row = tickets[tickets.ticket_id == ticket_id]
    if row.empty:
        raise KeyError(f"no ticket {ticket_id}")
    return row.iloc[0].to_json()


TOOLS = {"search_tickets": search_tickets, "ticket_counts": ticket_counts, "get_ticket": get_ticket}


# Used by the last check: one full run of your pipeline.
def run_pipeline():
    TRACE.clear()
    findings, sources = orchestrate(["charged twice refund", "parcel not delivered", "app crashes cart"])
    counts = call("ticket_counts", {"field": "category"})
    report = write_report(findings, counts)
    review(report, sources)
    call("ticket_counts", {"field": "category"})   # the orchestrator asks twice: a wasted repeat
    return trace_summary(TRACE)
print("Ready! Data loaded.")

## Milestone 1: Describe the tools (tools/list)

An MCP server tells clients what tools it has. Write `list_tools()` returning one entry per function in `TOOLS` (in that order):
`{"name": …, "description": first line of the docstring, "inputSchema": {"type": "object", "properties": {param: {"type": …}}, "required": [params without defaults]}}` — with Python types mapped str → "string", int → "integer".

<details>
<summary><b>💡 Hint</b></summary>

`inspect.signature(fn).parameters.values()` gives each parameter's `.annotation` and `.default` (which is `inspect.Parameter.empty` when there is none). `inspect.getdoc(fn)` gives the docstring.

</details>

In [ ]:
def list_tools():
    raise NotImplementedError

In [ ]:
check("Describe the tools (tools/list)", lambda: list_tools(), [{'name': 'search_tickets', 'description': 'Search open tickets by keywords; most matching words first.', 'inputSchema': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'integer'}}, 'required': ['query']}}, {'name': 'ticket_counts', 'description': 'Count open tickets by a field: product, category or priority.', 'inputSchema': {'type': 'object', 'properties': {'field': {'type': 'string'}}, 'required': ['field']}}, {'name': 'get_ticket', 'description': 'Full details of one ticket.', 'inputSchema': {'type': 'object', 'properties': {'ticket_id': {'type': 'string'}}, 'required': ['ticket_id']}}], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def list_tools():
    types = {str: "string", int: "integer", float: "number", bool: "boolean"}
    out = []
    for name, fn in TOOLS.items():
        params = list(inspect.signature(fn).parameters.values())
        out.append({"name": name, "description": inspect.getdoc(fn).splitlines()[0],
                    "inputSchema": {"type": "object",
                                    "properties": {p.name: {"type": types[p.annotation]} for p in params},
                                    "required": [p.name for p in params if p.default is inspect.Parameter.empty]}})
    return out
```

</details>

## Milestone 2: The server: handle JSON-RPC messages

Write `handle(raw)` — the heart of an MCP server. It receives one JSON-RPC message as text and returns the reply as text, or `None` for notifications (no `id`):
- invalid JSON → error `-32700`; `initialize` → result `{"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}, "serverInfo": {"name": "tickets", "version": "1.0"}}`
- `tools/list` → `{"tools": list_tools()}`
- `tools/call` → `{"content": [{"type": "text", "text": <tool output>}], "isError": False}`; if the tool raises, the same shape with the error message and `"isError": True`
- anything else → error `-32601` with message `"Method not found"`.

Replies look like `{"jsonrpc": "2.0", "id": …, "result": …}` or `{"jsonrpc": "2.0", "id": …, "error": {"code": …, "message": …}}` (use id `None` when the JSON couldn't be parsed).

In [ ]:
def handle(raw):
    raise NotImplementedError

In [ ]:
check("The server: handle JSON-RPC messages", lambda: [json.loads(handle('{"jsonrpc":"2.0","id":1,"method":"initialize","params":{}}'))["result"]["protocolVersion"], handle('{"jsonrpc":"2.0","method":"notifications/initialized"}'), len(json.loads(handle('{"jsonrpc":"2.0","id":2,"method":"tools/list"}'))["result"]["tools"]), json.loads(handle(json.dumps({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "ticket_counts", "arguments": {"field": "priority"}}})))["result"], json.loads(handle(json.dumps({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "ticket_counts", "arguments": {"field": "colour"}}})))["result"]["isError"], json.loads(handle('{"jsonrpc":"2.0","id":5,"method":"tools/delete"}'))["error"]["code"], json.loads(handle("{oops"))["error"]["code"]], ['2025-06-18', None, 3, {'content': [{'type': 'text', 'text': '{"high": 26, "low": 34, "medium": 55}'}], 'isError': False}, True, -32601, -32700], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def handle(raw):
    try:
        msg = json.loads(raw)
    except json.JSONDecodeError:
        return json.dumps({"jsonrpc": "2.0", "id": None, "error": {"code": -32700, "message": "Parse error"}})
    if "id" not in msg:
        return None
    method, params = msg.get("method"), msg.get("params", {})
    if method == "initialize":
        result = {"protocolVersion": "2025-06-18", "capabilities": {"tools": {}}, "serverInfo": {"name": "tickets", "version": "1.0"}}
    elif method == "tools/list":
        result = {"tools": list_tools()}
    elif method == "tools/call":
        try:
            text, is_error = TOOLS[params["name"]](**params.get("arguments", {})), False
        except Exception as e:
            text, is_error = f"{type(e).__name__}: {e}", True
        result = {"content": [{"type": "text", "text": text}], "isError": is_error}
    else:
        return json.dumps({"jsonrpc": "2.0", "id": msg["id"], "error": {"code": -32601, "message": "Method not found"}})
    return json.dumps({"jsonrpc": "2.0", "id": msg["id"], "result": result})
```

</details>

## Milestone 3: The client: call tools over the protocol

Agents never touch `TOOLS` directly; they go through the protocol, exactly as they would with a remote server. Write `call(name, arguments)`:
1. build a `tools/call` request with a new id each time (use the global counter `NEXT_ID`, starting at 100),
2. send its JSON text to `handle`, parse the reply,
3. return the parsed JSON of the tool's text output — or raise `RuntimeError(text)` if `isError` is true.

Also record every call in the list `TRACE` as `(name, arguments)`; you'll use it later.

In [ ]:
NEXT_ID = 100
TRACE = []


def call(name, arguments):
    raise NotImplementedError

In [ ]:
check("The client: call tools over the protocol", lambda: [call('ticket_counts', {'field': 'category'}), [t['id'] for t in call('search_tickets', {'query': 'refund not received', 'limit': 3})]], [{'billing': 32, 'bug': 34, 'delivery': 28, 'login': 21}, ['T113', 'T118', 'T150']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
NEXT_ID = 100
TRACE = []


def call(name, arguments):
    global NEXT_ID
    NEXT_ID += 1
    request = {"jsonrpc": "2.0", "id": NEXT_ID, "method": "tools/call", "params": {"name": name, "arguments": arguments}}
    TRACE.append((name, arguments))
    reply = json.loads(handle(json.dumps(request)))
    block = reply["result"]
    text = block["content"][0]["text"]
    if block["isError"]:
        raise RuntimeError(text)
    return json.loads(text)
```

</details>

## Milestone 4: Research agents, fanned out

The orchestrator gives each research agent one topic. A research agent searches tickets for its topic (limit 5) and returns `{"topic": …, "count": number found, "high": ids of high-priority matches, "sources": all ids found}`.

Write `research(topic)` and `orchestrate(topics)`, which runs `research` for every topic and returns `(list of findings, all sources de-duplicated in first-seen order)`.

<details>
<summary><b>💡 Hint</b></summary>

`dict.fromkeys(...)` de-duplicates while keeping order.

</details>

In [ ]:
def research(topic):
    raise NotImplementedError


def orchestrate(topics):
    raise NotImplementedError

In [ ]:
check("Research agents, fanned out", lambda: orchestrate(['charged twice refund', 'parcel not delivered', 'app crashes cart']), [[{'topic': 'charged twice refund', 'count': 5, 'high': ['T011', 'T167', 'T177'], 'sources': ['T011', 'T012', 'T029', 'T167', 'T177']}, {'topic': 'parcel not delivered', 'count': 5, 'high': ['T005', 'T041', 'T101'], 'sources': ['T005', 'T033', 'T041', 'T084', 'T101']}, {'topic': 'app crashes cart', 'count': 5, 'high': ['T015', 'T062'], 'sources': ['T013', 'T015', 'T059', 'T062', 'T088']}], ['T011', 'T012', 'T029', 'T167', 'T177', 'T005', 'T033', 'T041', 'T084', 'T101', 'T013', 'T015', 'T059', 'T062', 'T088']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def research(topic):
    found = call("search_tickets", {"query": topic, "limit": 5})
    return {"topic": topic, "count": len(found), "high": [t["id"] for t in found if t["priority"] == "high"],
            "sources": [t["id"] for t in found]}


def orchestrate(topics):
    findings = [research(t) for t in topics]
    sources = list(dict.fromkeys(s for f in findings for s in f["sources"]))
    return findings, sources
```

</details>

Here the agents run one after another. Because they're independent, a real orchestrator runs them in parallel (`asyncio.gather` or a thread pool), so total time is the slowest agent, not the sum.

## Milestone 5: A writer agent with citations

Write `write_report(findings, counts)` that returns the Monday report as text:
- first line: `"Open tickets by category: "` followed by `category n` pairs from `counts`, joined with `", "`,
- then one line per finding: `"- <topic>: <count> open tickets, <k> high priority [T001, T002]"`, citing that finding's high-priority ids (or all its sources if there are none).

Every claim must carry its sources — that's what lets a reviewer (and the support head) check it.

In [ ]:
def write_report(findings, counts):
    raise NotImplementedError

In [ ]:
check("A writer agent with citations", lambda: write_report(*[orchestrate(['charged twice refund', 'parcel not delivered'])[0]], call('ticket_counts', {'field': 'category'})).splitlines(), ['Open tickets by category: billing 32, bug 34, delivery 28, login 21', '- charged twice refund: 5 open tickets, 3 high priority [T011, T167, T177]', '- parcel not delivered: 5 open tickets, 3 high priority [T005, T041, T101]'], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def write_report(findings, counts):
    lines = ["Open tickets by category: " + ", ".join(f"{k} {v}" for k, v in counts.items())]
    for f in findings:
        cite = f["high"] or f["sources"]
        lines.append(f"- {f['topic']}: {f['count']} open tickets, {len(f['high'])} high priority [{', '.join(cite)}]")
    return "\n".join(lines)
```

</details>

## Milestone 6: A reviewer agent that rejects unsupported claims

Write `review(report, allowed_ids)` returning a sorted list of problems:
- `"uncited: <line>"` for every line starting with `"- "` that has no `[T…]` citation,
- `"unknown source: T…"` for every cited id not in `allowed_ids` (a hallucinated source).

An empty list means the report can be published. The check reviews your real report and a doctored one.

<details>
<summary><b>💡 Hint</b></summary>

`re.findall(r'T\d{3}', line)` finds ids.

</details>

In [ ]:
def review(report, allowed_ids):
    raise NotImplementedError

In [ ]:
check("A reviewer agent that rejects unsupported claims", lambda: (lambda f, s: [review(write_report(f, call('ticket_counts', {'field': 'category'})), s), review("Top issue\n- refunds are slow\n- login fails [T999]", s)])(*orchestrate(['charged twice refund', 'parcel not delivered'])), [[], ['uncited: - refunds are slow', 'unknown source: T999']], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def review(report, allowed_ids):
    problems = set()
    allowed = set(allowed_ids)
    for line in report.splitlines():
        ids = re.findall(r"T\d{3}", line)
        if line.startswith("- ") and not ids:
            problems.add(f"uncited: {line}")
        problems.update(f"unknown source: {i}" for i in ids if i not in allowed)
    return sorted(problems)
```

</details>

## Milestone 7: What did the agents do? A trace summary

Write `trace_summary(trace)` returning `{"calls": total tool calls, "by_tool": {tool: count} sorted by tool name, "repeated": number of calls that exactly repeat an earlier (tool, arguments)}`. Repeats are wasted work — a sign an agent should cache or remember results.

The check resets `TRACE`, runs one full pipeline, and summarises it.

<details>
<summary><b>💡 Hint</b></summary>

Turn arguments into a hashable key with `json.dumps(args, sort_keys=True)`.

</details>

In [ ]:
def trace_summary(trace):
    raise NotImplementedError

In [ ]:
check("What did the agents do? A trace summary", lambda: run_pipeline(), {'calls': 5, 'by_tool': {'search_tickets': 3, 'ticket_counts': 2}, 'repeated': 1}, tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def trace_summary(trace):
    keys = [(name, json.dumps(args, sort_keys=True)) for name, args in trace]
    return {"calls": len(trace), "by_tool": dict(sorted(Counter(n for n, _ in trace).items())),
            "repeated": len(keys) - len(set(keys))}
```

</details>

## Run it as a real MCP server

The official Python SDK does the protocol work you just wrote by hand. Save this as `tickets_server.py` next to the CSV, install with `pip install mcp pandas`, and connect it to Claude Code with `claude mcp add tickets -- python tickets_server.py` (or add it to Claude Desktop's config). SDK import paths differ between versions — check the docs for the one you install.

```python
import json, pandas as pd
from mcp.server.fastmcp import FastMCP     # SDK v1; v2 renames it to MCPServer (mcp.server.mcpserver)

mcp = FastMCP("tickets")
tickets = pd.read_csv("support_tickets.csv")

@mcp.tool()
def ticket_counts(field: str) -> str:
    """Count open tickets by a field: product, category or priority."""
    return json.dumps(tickets[tickets.status == "open"][field].value_counts().to_dict())

if __name__ == "__main__":
    mcp.run()
```

## Optional: let Claude drive the tools

Replace the scripted agents with a real model: give Claude the tools (converted from your `list_tools()`) and run the agent loop, sending every tool call through your MCP client. Needs an Anthropic API key and costs a few cents.

In [ ]:
import os, importlib.util
if os.environ.get("ANTHROPIC_API_KEY") and importlib.util.find_spec("anthropic"):
    import anthropic
    client = anthropic.Anthropic()
    claude_tools = [{"name": t["name"], "description": t["description"], "input_schema": t["inputSchema"]} for t in list_tools()]
    messages = [{"role": "user", "content": "Which issues should engineering look at first this week? Cite ticket ids."}]
    for _ in range(8):
        reply = client.messages.create(model="claude-opus-5-5", max_tokens=1500, tools=claude_tools, messages=messages)
        messages.append({"role": "assistant", "content": reply.content})
        if reply.stop_reason != "tool_use":
            break
        results = []
        for block in reply.content:
            if block.type == "tool_use":
                try:
                    out = json.dumps(call(block.name, block.input))
                except RuntimeError as e:
                    out = str(e)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})
        messages.append({"role": "user", "content": results})
    print("".join(b.text for b in reply.content if b.type == "text"))
else:
    print("No API key or anthropic package — the scripted agents above already show the full pipeline.")

## Present your work

Write a short design document (your README):

1. **Architecture diagram**: MCP server ↔ client ↔ orchestrator → research agents → writer → reviewer, with tracing.
2. **The tools**: names, schemas and why each exists.
3. **The Monday report** your pipeline produced, and the reviewer's verdict.
4. **Trace**: how many tool calls, and the wasted repeat — how you'd remove it (caching, shared memory).
5. **Security**: read-only tools, no destructive operations, tool output treated as data (prompt injection).
6. **Next steps**: real Claude agents, parallel research, an evaluation set of weekly questions.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built an MCP server exposing a support-ticket database as typed tools over JSON-RPC, usable from Claude Desktop and Claude Code
- Designed a multi-agent pipeline (orchestrator, research, writer and reviewer agents) producing cited weekly reports, with a reviewer that rejects uncited or hallucinated sources
- Added tracing of every tool call, finding **X%** redundant calls, and an optional Claude agent loop driving the same MCP tools

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. Why put the tools behind MCP instead of calling Python functions directly?
2. Walk me through what happens on the wire when an agent calls a tool.
3. How does your reviewer stop hallucinated citations? What can it still miss?
4. When is multi-agent worth it here, and when would one agent be better?
5. How would you secure this server before exposing it to other teams?
6. How would you evaluate the weekly report's quality over time?

## Take it further

- Run the research agents in parallel with asyncio and measure the speed-up.
- Add a `resources` endpoint that serves the weekly report itself by URI.
- Cache tool results in the client to remove repeated calls.
- Build an eval set of 20 weekly questions with expected ticket ids and score the pipeline.